##### Copyright 2025 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Working with Images using OpenAI SDK and Gemini

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/openai/Working_with_Images.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

This notebook shows you how to use images in your prompts with Gemini using the **OpenAI SDK**. You'll learn how to:

- Use images from URLs
- Use local images (base64 encoded)
- Send multiple images in one prompt
- Combine images with text
- Ask questions about specific parts of images

## Setup

In [ ]:
%pip install -U -q openai pillow requests

### Initialize the OpenAI Client

In [ ]:
from openai import OpenAI
import os

try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except:
    GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY', '--enter-your-API-key-here--')

client = OpenAI(
    api_key=GOOGLE_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

MODEL = "gemini-2.5-flash"

## Method 1: Images from URLs

The simplest way to use images is to provide a URL:

In [ ]:
# Example: Analyze an image from a URL
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg"

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "What's in this image? Describe it in detail."},
                {
                    "type": "image_url",
                    "image_url": {"url": image_url}
                }
            ]
        }
    ],
    max_tokens=300
)

print(response.choices[0].message.content)

## Method 2: Local Images (Base64 Encoding)

For local images, encode them as base64:

In [ ]:
import base64
import requests
from PIL import Image
from io import BytesIO

# Helper function to encode images
def encode_image_from_url(url):
    """Download and encode an image from URL"""
    response = requests.get(url)
    return base64.b64encode(response.content).decode('utf-8')

def encode_image_from_file(filepath):
    """Encode a local image file"""
    with open(filepath, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# Example: Use base64 encoded image
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg"
base64_image = encode_image_from_url(image_url)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "What's the main color palette in this image?"},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    }
                }
            ]
        }
    ]
)

print(response.choices[0].message.content)

## Multiple Images in One Prompt

You can include multiple images in a single request:

In [ ]:
# Multiple images example
image1_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg"
image2_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Placeholder_view_vector.svg/681px-Placeholder_view_vector.svg.png"

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Compare these two images. What are the main differences?"},
                {"type": "image_url", "image_url": {"url": image1_url}},
                {"type": "image_url", "image_url": {"url": image2_url}}
            ]
        }
    ]
)

print(response.choices[0].message.content)

## Interleaving Text and Images

You can mix text and images in various orders:

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "I have an image of a landscape."},
                {"type": "image_url", "image_url": {"url": image1_url}},
                {"type": "text", "text": "Could you write a short poem inspired by this scene?"}
            ]
        }
    ]
)

print(response.choices[0].message.content)

## Asking Specific Questions About Images

Ask detailed questions about image content:

In [ ]:
questions = [
    "What objects can you identify in this image?",
    "What's the weather like in this scene?",
    "What time of day does this appear to be?",
    "Are there any people or animals visible?"
]

for question in questions:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": question},
                    {"type": "image_url", "image_url": {"url": image1_url}}
                ]
            }
        ],
        max_tokens=100
    )
    
    print(f"Q: {question}")
    print(f"A: {response.choices[0].message.content}")
    print()

## Image Analysis with Structured Output

Combine image analysis with JSON mode for structured results:

In [ ]:
import json

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": """Analyze this image and return a JSON object with:
                    - scene: brief description
                    - objects: list of main objects
                    - colors: dominant colors
                    - mood: overall mood/atmosphere"""
                },
                {"type": "image_url", "image_url": {"url": image1_url}}
            ]
        }
    ],
    response_format={"type": "json_object"}
)

analysis = json.loads(response.choices[0].message.content)
print(json.dumps(analysis, indent=2))

## Multi-turn Conversation with Images

Build a conversation about an image:

In [ ]:
# Start conversation with image
messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "What's in this image?"},
            {"type": "image_url", "image_url": {"url": image1_url}}
        ]
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages
)

print("User: What's in this image?")
print(f"Assistant: {response.choices[0].message.content}")
print()

# Add response to conversation
messages.append({
    "role": "assistant",
    "content": response.choices[0].message.content
})

# Continue conversation
messages.append({
    "role": "user",
    "content": "What would be a good title for this photo?"
})

response = client.chat.completions.create(
    model=MODEL,
    messages=messages
)

print("User: What would be a good title for this photo?")
print(f"Assistant: {response.choices[0].message.content}")

## Image Detail Level

Control the detail level of image analysis:

In [ ]:
# High detail analysis
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Provide a very detailed description of this image."},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image1_url,
                        "detail": "high"  # Options: "low", "high", "auto"
                    }
                }
            ]
        }
    ],
    max_tokens=500
)

print("High detail analysis:")
print(response.choices[0].message.content)

## Best Practices

1. **Image Size**: Keep images under 20MB for inline base64 encoding
2. **Format**: JPEG, PNG, GIF, and WebP are supported
3. **Multiple Images**: You can include multiple images, but be mindful of token limits
4. **URLs**: Use publicly accessible URLs or base64 encoding for private images
5. **Detail Level**: Use "low" detail for faster processing when high detail isn't needed

## Limitations

- **Video**: Not supported via OpenAI SDK. Use [Gemini SDK](../Video_understanding.ipynb) for video.
- **Large Files**: Files over 20MB should use the [Gemini File API](../File_API.ipynb).

## Next Steps

- [Structured Outputs](./Structured_Outputs.ipynb) - Extract structured data from images
- [Function Calling](./Function_Calling.ipynb) - Use tools with image analysis
- [Full OpenAI Compatibility Guide](../Get_started_OpenAI_Compatibility.ipynb)
- [Video Understanding](../Video_understanding.ipynb) - Process videos with Gemini SDK